# 10 · Subgraphs: Building Big Graphs from Small Ones

> **Kernel:** `Phase 5 · LangGraph` · **Runs offline**: the nodes are plain
> functions, so the focus stays on how graphs compose.

Real systems grow: a research team, a writing team, a review step, each with
its own internal steps. Putting everything in one flat graph makes it
unreadable and untestable. **Subgraphs** let you build each part as its own
graph, test it alone, and plug it into a bigger graph as a single node.

**You will learn**
- Two ways to use a subgraph: **as a node** (shared keys) or **called from a node** (different schema)
- How state flows in and out, and how to keep a subgraph's working keys private
- Persistence and **interrupts inside subgraphs**, and inspecting subgraph state
- When to split a graph, and when not to

**Prerequisites:** notebooks 02 (state, input/output schemas), 07 (interrupts),
08 (checkpoints).

In [ ]:
import sys, pathlib
SECTION = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
               if (p / 'common.py').exists())          # the 01_LANGGRAPH folder
sys.path.insert(0, str(SECTION))

import operator
from typing import TypedDict, Annotated
from langgraph.graph import StateGraph, START, END
from langgraph.types import interrupt, Command
from langgraph.checkpoint.memory import InMemorySaver

## 1 · Why subgraphs

| Benefit | What it means |
|---|---|
| **Modularity** | each part has one job and a small, readable graph |
| **Encapsulation** | a subgraph's working keys (scratchpads, retries) stay inside it |
| **Reuse** | the same "review" subgraph can serve several parent graphs |
| **Testing** | invoke a subgraph alone with a hand-made input |
| **Ownership** | different people can build different subgraphs against an agreed interface |

You've already used a subgraph without noticing: `create_agent` returns a
compiled graph, and in notebook 06 you added agents as nodes of a bigger graph.

## 2 · Way A: a compiled subgraph *is* a node

If the subgraph's state **shares keys** with the parent, you can add the
compiled subgraph directly with `add_node`. The shared keys are the interface:

```
 parent state:   topic, findings, report
                    │       ▲
                    ▼       │   (shared keys flow in and out)
 subgraph state: topic, findings, queries, drafts   ← queries/drafts: private working keys
```

The subgraph below has its own **input/output schemas** (notebook 02 §7), so
only `topic` goes in and only `findings` comes out, while `queries` stays private.

In [ ]:
# ---- the research subgraph -------------------------------------------------
class ResearchIn(TypedDict):
    topic: str

class ResearchOut(TypedDict):
    findings: list[str]

class ResearchState(TypedDict):                 # internal: includes private working keys
    topic: str
    queries: list[str]
    findings: list[str]


def make_queries(state: ResearchState) -> dict:
    return {'queries': [f"{state['topic']} {angle}" for angle in ('costs', 'risks', 'timeline')]}

def search(state: ResearchState) -> dict:
    return {'findings': [f'found facts on "{q}"' for q in state['queries']]}


r = StateGraph(ResearchState, input_schema=ResearchIn, output_schema=ResearchOut)
r.add_node('make_queries', make_queries)
r.add_node('search', search)
r.add_edge(START, 'make_queries')
r.add_edge('make_queries', 'search')
r.add_edge('search', END)
research_team = r.compile()

# A subgraph is a graph: test it on its own.
print('alone:', research_team.invoke({'topic': 'Utrecht rollout'}))

In [ ]:
# ---- the parent graph --------------------------------------------------------
class Project(TypedDict):
    topic: str
    findings: list[str]
    report: str


def write_report(state: Project) -> dict:
    return {'report': f"Report on {state['topic']}: " + '; '.join(state['findings'])}


p = StateGraph(Project)
p.add_node('research', research_team)            # ← the compiled subgraph, as one node
p.add_node('write_report', write_report)
p.add_edge(START, 'research')
p.add_edge('research', 'write_report')
p.add_edge('write_report', END)
project = p.compile()

out = project.invoke({'topic': 'Utrecht rollout'})
print(out['report'])
print('keys in parent state:', list(out), ' ← no "queries": it stayed inside the subgraph')

The parent sees `research` as one step. Its internal steps are visible only if
you ask (`stream(..., subgraphs=True)`, notebook 09). Its private `queries`
never leak into the parent's state.

## 3 · Way B: call a subgraph from inside a node

Sometimes the subgraph's schema has **nothing in common** with the parent, for
example a reusable "translator" graph with keys `source_text` / `translated`. Then
wrap it in a normal node that **maps** parent state to subgraph input and the
subgraph's output back to a parent update:

In [ ]:
class TranslateState(TypedDict):
    source_text: str
    language: str
    translated: str

def translate(state: TranslateState) -> dict:
    fake = {'nl': 'Robots bezorgen boodschappen.', 'hi': 'रोबोट किराने का सामान पहुँचाते हैं।'}
    return {'translated': fake.get(state['language'], state['source_text'])}

t = StateGraph(TranslateState)
t.add_node('translate', translate)
t.add_edge(START, 'translate')
translator = t.compile()


class Release(TypedDict):
    headline: str
    localized: Annotated[dict, lambda old, new: {**(old or {}), **new}]


def localize_dutch(state: Release) -> dict:
    result = translator.invoke({'source_text': state['headline'], 'language': 'nl'})   # map IN
    return {'localized': {'nl': result['translated']}}                                  # map OUT

def localize_hindi(state: Release) -> dict:
    result = translator.invoke({'source_text': state['headline'], 'language': 'hi'})
    return {'localized': {'hi': result['translated']}}


p = StateGraph(Release)
p.add_node('localize_dutch', localize_dutch)
p.add_node('localize_hindi', localize_hindi)
p.add_edge(START, 'localize_dutch')              # the same subgraph, used twice, in parallel
p.add_edge(START, 'localize_hindi')
release = p.compile()
print(release.invoke({'headline': 'Robots deliver groceries.', 'localized': {}}))

| | Way A: subgraph as a node | Way B: call it inside a node |
|---|---|---|
| Schemas | must share the keys it reads/writes | can be completely different |
| Mapping | automatic, via shared keys | you write it (in and out) |
| Visible in the parent's graph drawing | yes, as a node | as an ordinary node |
| Typical use | teams inside one system | reusable components with their own vocabulary |

Both are real subgraphs at runtime: both stream with `subgraphs=True` and both
checkpoint under the parent's thread.

## 4 · Persistence and interrupts inside a subgraph

You compile the **parent** with a checkpointer. Subgraphs automatically use
it, stored under their own **namespace** within the parent's thread. So an
`interrupt()` deep inside a subgraph pauses the **whole** run, and resuming the
parent resumes the subgraph exactly where it stopped.

In [ ]:
class ReviewState(TypedDict):
    draft: str
    verdict: str

def ask_reviewer(state: ReviewState) -> dict:
    answer = interrupt({'please_review': state['draft']})     # pauses inside the SUBGRAPH
    return {'verdict': answer}

rv = StateGraph(ReviewState)
rv.add_node('ask_reviewer', ask_reviewer)
rv.add_edge(START, 'ask_reviewer')
review_team = rv.compile()                                     # no checkpointer here


class Pipeline(TypedDict):
    draft: str
    verdict: str
    status: str

def draft(state):   return {'draft': 'Utrecht go-live moves to 31 October.'}
def finish(state):  return {'status': f"done, reviewer said {state['verdict']!r}"}

p = StateGraph(Pipeline)
p.add_node('draft', draft)
p.add_node('review', review_team)                              # subgraph as a node
p.add_node('finish', finish)
p.add_edge(START, 'draft')
p.add_edge('draft', 'review')
p.add_edge('review', 'finish')
pipeline = p.compile(checkpointer=InMemorySaver())             # ONE checkpointer, on the parent

cfg = {'configurable': {'thread_id': 'pipe-1'}}
paused = pipeline.invoke({'draft': ''}, cfg)
print('paused      :', paused['__interrupt__'][0].value)

snap = pipeline.get_state(cfg, subgraphs=True)
print('parent next :', snap.next)
print('inside it   :', snap.tasks[0].state.next, '← the subgraph is waiting at this node')

print('resumed     :', pipeline.invoke(Command(resume='approved'), cfg)['status'])

`get_state(cfg, subgraphs=True)` lets you look *inside*: the parent is waiting
at `review`, and within the subgraph the waiting node is `ask_reviewer`.

**Subgraph memory across calls.** By default a subgraph starts fresh each time
the parent calls it. If a subgraph needs its **own** memory that persists across
the parent's calls (e.g. a sub-agent that should remember its own earlier
conversation within the thread), compile it with `checkpointer=True`. Use this
sparingly: most subgraphs should be stateless between calls.

## 5 · When to split a graph

**Split** when:
- a part has its own internal loop or branching that clutters the parent
- a part has private working state
- you want to reuse or test a part on its own
- different people own different parts

**Don't split** when:
- the "subgraph" would be one or two nodes (just use nodes)
- the parts need to share lots of state back and forth (the interface becomes the whole state)

## 6 · Common mistakes

| Mistake | Symptom | Fix |
|---|---|---|
| Subgraph added as a node but shares no keys | it receives nothing useful / its output vanishes | share keys, or use way B with explicit mapping |
| Private keys without an output schema | working keys leak into the parent | give the subgraph `input_schema` / `output_schema` |
| A separate checkpointer on each subgraph | confusing, split persistence | put one checkpointer on the parent |
| Parallel subgraphs writing one parent key | `InvalidUpdateError` | a reducer on that key (as `localized` above) |
| Deep nesting "because it's tidy" | hard to trace and debug | keep nesting shallow, usually 1–2 levels |

## 7 · Check yourself

<details><summary><b>1.</b> What decides which data flows between a parent and a subgraph added directly as a node?</summary>

The keys the two state schemas share. With input/output schemas on the subgraph, only the keys in its input schema flow in and only those in its output schema flow out.
</details>

<details><summary><b>2.</b> When must you call a subgraph from inside a node instead?</summary>

When the schemas don't share keys (or you want to reshape data). The wrapper node maps the parent state to the subgraph's input and maps its output back to a parent update.
</details>

<details><summary><b>3.</b> An <code>interrupt()</code> fires inside a subgraph. Where is the pause saved, and how do you resume?</summary>

In the parent's checkpointer, under the subgraph's namespace within the same thread. Resume the parent with `Command(resume=...)` and the same thread id; the subgraph continues where it paused.
</details>

<details><summary><b>4.</b> How do you see which node a subgraph is waiting at?</summary>

`graph.get_state(config, subgraphs=True)`: the parent's snapshot includes tasks whose `state` is the subgraph's snapshot, with its own `next`.
</details>

<details><summary><b>5.</b> Give one good reason and one bad reason to turn part of a graph into a subgraph.</summary>

Good: it has private working state or its own loop, or it's reused or tested separately. Bad: it's only one or two nodes, or it would need to exchange almost the whole state with the parent.
</details>

## Takeaway

A compiled graph can be a node. Add it directly when it shares keys with the
parent (using input/output schemas to keep its working keys private), or call it
inside a node with explicit mapping when the schemas differ. One checkpointer
on the parent covers everything, including interrupts deep inside subgraphs.

Next → `11_mcp_tools.ipynb`: connecting LangGraph to tools served over the
Model Context Protocol.